# PCA From Scratch

Part of **Linear Algebra**.

## 1. Intuition First

PCA keeps the direction that preserves as much centered variance as possible when you compress the data to a line.

![PCA projection](../assets/diagrams/pca_projection.svg)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5)

## 2. Variance Maximization Derivation

Let the centered data matrix be $X_c \in \mathbb{R}^{n \times d}$ and let $u \in \mathbb{R}^d$ be a unit vector.
The variance of the projected dataset is

$$
\mathrm{Var}(X_c u) = \frac{1}{n} u^\top X_c^\top X_c u.
$$

If we define $\Sigma = \frac{1}{n} X_c^\top X_c$, then PCA solves

$$
\max_{\|u\|_2 = 1} u^\top \Sigma u.
$$

Using a Lagrange multiplier $\lambda$,

$$
\mathcal{L}(u, \lambda) = u^\top \Sigma u - \lambda (u^\top u - 1),
$$

and differentiating gives

$$
\Sigma u = \lambda u.
$$

So the first principal component is the eigenvector with the largest eigenvalue.

In [ ]:
from src.linear_algebra import pca_from_scratch

angles = np.linspace(-2.5, 2.5, 14)
X = np.column_stack((2.0 * angles + 0.25 * np.sin(angles), 1.1 * angles + 0.45 * np.cos(angles)))
result = pca_from_scratch(X, n_components=2)
centered = X - result.mean
first_axis = result.components[0]
line_coordinates = centered @ first_axis
projected = np.outer(line_coordinates, first_axis) + result.mean

print("principal axis =", first_axis)
print("explained variance ratio =", result.explained_variance_ratio)

## 3. PyTorch Verification

The SVD of the centered matrix gives the same principal directions as the covariance eigendecomposition.

In [ ]:
X_t = torch.tensor(X, dtype=torch.float64)
centered_t = X_t - X_t.mean(dim=0)
_, singular_values, Vh = torch.linalg.svd(centered_t, full_matrices=False)
torch_axis = Vh[0]

print(torch.allclose(torch.tensor(first_axis).abs(), torch_axis.abs(), atol=1e-6))
print("singular values =", singular_values)

## 4. Custom Figure

The dashed connectors show the orthogonal projection of each point onto the first component.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[:, 0], X[:, 1], color="tab:blue", label="original points")
ax.scatter(projected[:, 0], projected[:, 1], color="tab:orange", label="projected points")
for original, target in zip(X, projected):
    ax.plot([original[0], target[0]], [original[1], target[1]], linestyle="--", color="#94a3b8", linewidth=1.5)
ax.quiver(result.mean[0], result.mean[1], first_axis[0], first_axis[1], angles="xy", scale_units="xy", scale=0.3, color="tab:red", width=0.007)
ax.set_title("PCA projects data onto a maximal-variance axis")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.legend()
plt.show()

## 5. Where This Shows Up

- data whitening before optimization
- low-rank approximations for embeddings and activations
- latent compression and visualization